<a href="https://colab.research.google.com/github/Vinit-Rabadiya/pmi-gene-expression-analysis/blob/main/pmi_gene_expression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import urllib.request
import gzip

print("--- STEP 1: DOWNLOADING DATASET METADATA FROM NCBI GEO ---")
# GSE86354 provides processed GTEx expression samples with metadata (PMI/ischemic time and RIN)
url_meta = "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE86354&format=file&file=GSE86354_GTEx_FPKM.samples.txt.gz"
output_file = "gse86354_samples.txt.gz"

urllib.request.urlretrieve(url_meta, output_file)
print("Download complete.")

# Unpack and load the metadata into a Pandas DataFrame
with gzip.open(output_file, 'rt') as f:
    df_meta = pd.read_csv(f, sep='\t')

print(f"\nSuccessfully loaded metadata! Total records: {len(df_meta)}")
print("Columns found in metadata:")
print(df_meta.columns.tolist())

# Display the first few rows to verify column names
display(df_meta.head(3))

--- STEP 1: DOWNLOADING DATASET METADATA FROM NCBI GEO ---
Download complete.

Successfully loaded metadata! Total records: 1557
Columns found in metadata:
['GTEX-U8XE-1526-SM-4E3HT', 'Liver']


,GTEX-U8XE-1526-SM-4E3HT,Liver
0,GTEX-RWSA-2026-SM-47JX8,Prostate
1,GTEX-X4XX-0926-SM-46MV7,Thyroid
2,GTEX-T6MO-1526-SM-4DM57,Uterus


In [2]:
print("\n--- STEP 2: TISSUE FILTERING & ATTRIBUTE EXTRACTION ---")

# Inspect available tissue types if a tissue column exists
tissue_col = [col for col in df_meta.columns if 'tissue' in col.lower() or 'site' in col.lower()]
print(f"Identified tissue column: {tissue_col}")

if tissue_col:
    t_col = tissue_col[0]
    print("Available tissues:", df_meta[t_col].unique())

    # Select target tissue agreed upon by the group (e.g., Whole Blood or Skin)
    target_tissue = 'Whole Blood'
    df_subset = df_meta[df_meta[t_col] == target_tissue].copy()
else:
    df_subset = df_meta.copy()

# Look for ischemic time and RIN columns
pmi_col = [c for c in df_meta.columns if 'trischd' in c.lower() or 'pmi' in c.lower() or ' ischemic' in c.lower()]
rin_col = [c for c in df_meta.columns if 'smrin' in c.lower() or 'rin' in c.lower()]

print(f"Ischemic time column candidates: {pmi_col}")
print(f"RIN column candidates: {rin_col}")

# Clean and filter out missing values for these metrics if columns are identified
if pmi_col and rin_col:
    p_col, r_col = pmi_col[0], rin_col[0]
    df_subset = df_subset.dropna(subset=[p_col, r_col])
    print(f"\nExtracted subset shape for modeling: {df_subset.shape}")
    print(df_subset[[p_col, r_col]].describe())
else:
    print("Please check column names above to map your target variables manually.")


--- STEP 2: TISSUE FILTERING & ATTRIBUTE EXTRACTION ---
Identified tissue column: []
Ischemic time column candidates: []
RIN column candidates: []
Please check column names above to map your target variables manually.
